# Scoring an AiDotNet Model in Databricks via ONNX

**Training session demo notebook** — the end-to-end demonstration for the Databricks intro session.

**What this notebook does:**
1. Installs `onnxruntime` on the cluster
2. Loads `telco_churn.onnx` (exported by AiDotNet's `ConvertToOnnx` path)
3. Reads `telco_churn_test_data.csv` as a Spark DataFrame and saves it as a Delta table (showcases Delta)
4. Defines a Spark `pandas_udf` that runs ONNX inference at scale
5. Applies the UDF, compares predictions to the expected outputs AiDotNet recorded
6. Logs the run to MLflow

**What the audience learns:**
- Databricks workspace + notebook navigation
- Spark DataFrames vs pandas
- Delta Lake basics (`saveAsTable`, `spark.read.table`)
- `pandas_udf` for vectorised Python at scale
- MLflow for run tracking
- How a model trained outside Databricks (here: in .NET, via AiDotNet) deploys for inference inside Databricks

## Step 1 — Install `onnxruntime` on the cluster

Databricks Community Edition clusters don't have `onnxruntime` pre-installed. The `%pip install` magic restarts the Python interpreter for this notebook so the new package is available.

In [ ]:
%pip install onnxruntime --quiet

## Step 2 — Upload the model and the test data

Before running this cell, upload `telco_churn.onnx` and `telco_churn_test_data.csv` to the workspace.

In Databricks Community Edition, the easiest path:
- Click **Catalog** → **Upload data** → choose both files → upload to `/FileStore/`.
- The files end up at `/dbfs/FileStore/telco_churn.onnx` and `/dbfs/FileStore/telco_churn_test_data.csv`.

Adjust the paths below if you uploaded somewhere else.

In [ ]:
MODEL_PATH = "/dbfs/FileStore/telco_churn.onnx"
CSV_PATH   = "/dbfs/FileStore/telco_churn_test_data.csv"
DELTA_TABLE = "telco_churn_test"

# Quick sanity check that the files are where we expect.
import os
for p in [MODEL_PATH, CSV_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Expected {p} — upload via Catalog → Upload data")
    print(f"✓ Found {p} ({os.path.getsize(p):,} bytes)")

## Step 3 — Load CSV as a Spark DataFrame and save to Delta

This is one of Databricks' core teaching moments: a Spark DataFrame and a Delta table are the same shape of data, but Delta gives you ACID transactions, time travel, and schema enforcement.

First we read the CSV. Then we **save it as a Delta table** with `saveAsTable`. From then on, anyone in the workspace can `SELECT * FROM telco_churn_test` with SQL.

In [ ]:
from pyspark.sql.functions import col

# Read the CSV. Header=true picks up our column names. inferSchema gives us
# float columns instead of string columns.
df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(CSV_PATH.replace("/dbfs", "dbfs:"))
)
df.printSchema()
df.show(5)
print(f"Total rows: {df.count():,}")

In [ ]:
# Save as a Delta table. The 'overwrite' mode lets you re-run this notebook.
(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable(DELTA_TABLE)
)

# Sanity: SQL works against the table now.
spark.sql(f"SELECT COUNT(*) AS row_count FROM {DELTA_TABLE}").show()

## Step 4 — Define a `pandas_udf` that runs ONNX inference

`pandas_udf` is the idiomatic way to run vectorised Python (NumPy, pandas, scikit-learn, ONNX Runtime…) at Spark scale.  It receives pandas Series as inputs, returns a pandas Series, and Spark runs it in parallel across the cluster.

We broadcast the model bytes so every worker gets its own `InferenceSession` lazily (avoiding the cost of pickling the session itself).

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import FloatType

# Read the model bytes once on the driver, broadcast to all workers.
with open(MODEL_PATH, "rb") as f:
    model_bytes = f.read()
broadcast_model = spark.sparkContext.broadcast(model_bytes)

# Lazy per-worker session; cached so it's only built once per executor.
_session_cache = {}

def _get_session():
    if "sess" not in _session_cache:
        import onnxruntime as ort
        _session_cache["sess"] = ort.InferenceSession(broadcast_model.value)
    return _session_cache["sess"]

@pandas_udf(FloatType())
def predict_churn(tenure: pd.Series, monthly: pd.Series, total: pd.Series, contract: pd.Series) -> pd.Series:
    sess = _get_session()
    X = np.column_stack([tenure.values, monthly.values, total.values, contract.values]).astype(np.float32)
    preds = sess.run(None, {"input": X})[0].flatten().astype(np.float32)
    return pd.Series(preds)

## Step 5 — Apply the UDF to the Delta table

In [ ]:
scored = (
    spark.table(DELTA_TABLE)
         .withColumn(
             "onnx_prediction",
             predict_churn(col("tenure_norm"), col("monthly_norm"), col("total_norm"), col("contract_norm")),
         )
)
scored.select("tenure_norm", "contract_norm", "churn_actual", "churn_predicted", "onnx_prediction").show(10)

## Step 6 — Verify parity with AiDotNet

`churn_predicted` is what AiDotNet recorded when it exported the model. `onnx_prediction` is what we just computed inside Spark via ONNX Runtime. If everything works, the two columns match to within ~1e-5.

This is the moment that proves the round-trip: AiDotNet trained the model in .NET, exported to ONNX, and Databricks scores it identically at scale.

In [ ]:
from pyspark.sql.functions import abs as sql_abs

diffs = (
    scored
      .withColumn("abs_diff", sql_abs(col("churn_predicted") - col("onnx_prediction")))
      .agg(
          {"abs_diff": "max", "abs_diff": "avg"}
      )
)
diffs.show()

mismatches = scored.filter(
    sql_abs(col("churn_predicted") - col("onnx_prediction")) > 1e-3
).count()
total = scored.count()
print(f"Mismatches above 1e-3: {mismatches} / {total}")
assert mismatches == 0, "ONNX Runtime predictions differ from AiDotNet by more than 1e-3 — check the model upload and inputs."

## Step 7 — Log the run to MLflow

MLflow is built into Databricks. Logging the model + score-time metrics gives you a permanent record of *which* model produced *which* predictions on *which* dataset.

In [ ]:
import mlflow

with mlflow.start_run(run_name="telco-churn-onnx-scoring"):
    mlflow.log_param("model_source",        "AiDotNet (C#) via ConvertToOnnx")
    mlflow.log_param("model_file",          MODEL_PATH)
    mlflow.log_param("model_bytes",         len(model_bytes))
    mlflow.log_param("input_table",         DELTA_TABLE)
    mlflow.log_metric("rows_scored",        scored.count())
    mlflow.log_metric("mismatches_1e_minus_3", mismatches)
    mlflow.log_artifact(MODEL_PATH)
    print("Logged to MLflow — see the Experiments tab in the left sidebar.")

## Wrap-up

You just:
- Loaded an ONNX model that was trained in .NET (via AiDotNet)
- Stored test data as a Delta table
- Defined a `pandas_udf` to run inference at Spark scale
- Confirmed the in-Databricks predictions match the AiDotNet-recorded predictions to within 1e-3
- Logged the whole run to MLflow

**The teaching point:** any model you can export to ONNX (from PyTorch, TensorFlow, ML.NET, AiDotNet, …) plugs into Databricks the same way. The model-authoring runtime is decoupled from the model-serving runtime.